# Перенос текущего SNO v2 на другую нумерацию или сетку

Этот notebook показывает API frozen P1-переноса, без сохранённых экспериментальных результатов. Среднее и POD-моды интерполируются по исходным треугольникам; веса и нормировка не переобучаются. Сначала проверяем точную перестановку узлов. Для новой физической триангуляции требуется отдельный FEM reference и проверка покрытия области.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'neural_operator').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
import numpy as np
import matplotlib.pyplot as plt
from neural_operator.data import load_dataset
from neural_operator.training import load_predictor, dataset_fingerprint
from neural_operator.plotting import display_table, plot_prediction
BASE = ROOT / 'neural_operator'
DATA_DIR = BASE / 'data' / 'static_low_frequency_1000x8'
MODEL_RUNS = {
    'svd': BASE / 'runs' / 'revision_2026-11-08' / 'static_low' / 'svd',
    'chebyshev': BASE / 'runs' / 'revision_2026-11-08' / 'static_low' / 'chebyshev',
    'fno': BASE / 'randomized' / 'runs' / 'low_frequency_static_1000x8' / 'fno',
}


In [ ]:
predictor = load_predictor(MODEL_RUNS['svd'] / 'model.pt', device='cpu')
dataset = load_dataset(DATA_DIR, mmap=True)
index = int(dataset.split_indices()[2][0])
source = predictor.predict_dataset(dataset, [index])[0]
permutation = np.random.default_rng(42).permutation(len(dataset.coordinates))
inverse = np.argsort(permutation)
coordinates = dataset.coordinates[permutation]
triangles = inverse[dataset.triangles]
mapped = predictor.on_mesh(coordinates, triangles)
prediction = mapped.predict_static(dataset.theta[index], dataset.k0[index])
np.testing.assert_allclose(prediction, source[permutation], rtol=2e-5, atol=2e-6)
print('Permutation check passed:', prediction.shape)


Для иной сетки передайте собственные `coordinates, triangles` в `predictor.on_mesh(...)`. `predict_static(theta,k0)` использует фиксированную сохранённую среду. Для меняющихся A,n нужны target triangles и `mapped.predict(A_target,n_target,theta,k0)`.

Модуль вычисляется после переноса Re/Im. Более тонкая target-сетка не добавляет пространственных мод обученному декодеру. Сравнивать модель с target FEM нужно в consistent mass-норме целевой сетки; ошибка переноса и изменение FEM-дискретизации измеряются отдельно. Этот шаблон не содержит измеренных cross-mesh результатов.
